# Ejemplo de cálculo de homografías a través de 5 pares de puntos

In [9]:
import numpy as np
import cv2

# Definimos 5 pares de puntos correspondientes (origen y destino)
# Funciona igual para exactamente 4 puntos o para N > 4
pts_origen = np.array([[0, 0], [100, 0], [100, 100], [0, 100], [50, 50]], dtype=np.float32)
pts_destino = np.array([[10, 20], [80, 10], [90, 90], [20, 80], [55, 45]], dtype=np.float32)

## Sin usar OpenCV

In [10]:
# Función auxiliar para la normalización de Hartley
def normalizacion_hartley(pts):
    # 1. Trasladar el centroide al origen
    centroide = np.mean(pts, axis=0)
    
    # 2. Calcular la distancia media al centroide
    distancias = np.linalg.norm(pts - centroide, axis=1)
    escala = np.sqrt(2) / np.mean(distancias)
    
    # 3. Construir la matriz de transformación de similitud T
    T = np.array([
        [escala, 0, -escala * centroide[0]],
        [0, escala, -escala * centroide[1]],
        [0, 0, 1]
    ], dtype=np.float32)
    
    # 4. Aplicar T a los puntos (usando coordenadas homogéneas)
    pts_homogeneos = np.column_stack((pts, np.ones(len(pts))))
    pts_norm = (T @ pts_homogeneos.T).T[:, :2] # Regresamos a 2D
    
    return pts_norm, T

# --- Inicio del cálculo robusto ---

# 1. Aplicar normalización de Hartley a ambos conjuntos de puntos
pts_origen_norm, T_origen = normalizacion_hartley(pts_origen)
pts_destino_norm, T_destino = normalizacion_hartley(pts_destino)

# 2. Construir la matriz A (2N x 9) con los puntos NORMALIZADOS
A = []
for (x, y), (xp, yp) in zip(pts_origen_norm, pts_destino_norm):
    A.append([-x, -y, -1, 0, 0, 0, x*xp, y*xp, xp])
    A.append([0, 0, 0, -x, -y, -1, x*yp, y*yp, yp])
A = np.array(A)

# 3. Resolver el sistema Ah = 0 usando SVD
_, _, Vt = np.linalg.svd(A)
H_norm = Vt[-1].reshape(3, 3)

# 4. Desnormalización geométrica: H = inv(T_destino) * H_norm * T_origen
H_numpy = np.linalg.inv(T_destino) @ H_norm @ T_origen

# 5. Normalizamos para que h33 = 1
H_numpy = H_numpy / H_numpy[2, 2]

np.set_printoptions(suppress=True)
print("Homografía pura con NumPy (SVD + Hartley):\n", H_numpy)

Homografía pura con NumPy (SVD + Hartley):
 [[ 0.55564525  0.08495737 11.9271991 ]
 [-0.08507909  0.62508222 17.36314206]
 [-0.00168835 -0.00012959  1.        ]]


## Con OpenCV

In [11]:
H_opencv, _ = cv2.findHomography(pts_origen, pts_destino, 0)

print("Homografía con OpenCV:\n", H_opencv)

# Comprobación de que ambas matrices son idénticas
error_maximo = np.max(np.abs(H_numpy - H_opencv))
print(f"\nDiferencia máxima entre ambos métodos: {error_maximo:.10f}")

Homografía con OpenCV:
 [[ 0.54604817  0.0809849  12.61685198]
 [-0.08131505  0.62681939 17.04925838]
 [-0.0016857  -0.00013969  1.        ]]

Diferencia máxima entre ambos métodos: 0.6896528851


## La discrepancia surge por cómo tratan los dos métodos que el sistema sea sobredeterminado
## Vamos a ver qué pasa solamente con 4 puntos

In [12]:
pts_origen_4 = pts_origen[:4]
pts_destino_4 = pts_destino[:4]

In [13]:
pts_origen_norm, T_origen = normalizacion_hartley(pts_origen_4)
pts_destino_norm, T_destino = normalizacion_hartley(pts_destino_4)

# 2. Construir la matriz A (2N x 9) con los puntos NORMALIZADOS
A = []
for (x, y), (xp, yp) in zip(pts_origen_norm, pts_destino_norm):
    A.append([-x, -y, -1, 0, 0, 0, x*xp, y*xp, xp])
    A.append([0, 0, 0, -x, -y, -1, x*yp, y*yp, yp])
A = np.array(A)

# 3. Resolver el sistema Ah = 0 usando SVD
_, _, Vt = np.linalg.svd(A)
H_norm = Vt[-1].reshape(3, 3)

# 4. Desnormalización geométrica: H = inv(T_destino) * H_norm * T_origen
H_numpy = np.linalg.inv(T_destino) @ H_norm @ T_origen

# 5. Normalizamos para que h33 = 1
H_numpy = H_numpy / H_numpy[2, 2]

np.set_printoptions(suppress=True)
print("Homografía pura con NumPy (SVD + Hartley):\n", H_numpy)

Homografía pura con NumPy (SVD + Hartley):
 [[ 0.49636362  0.10727272 10.00000102]
 [-0.12545455  0.62909089 20.00000077]
 [-0.00254545  0.00036364  1.        ]]


In [14]:
H_opencv, mascara = cv2.findHomography(pts_origen_4, pts_destino_4, 0)

print("Homografía con OpenCV:\n", H_opencv)

# Comprobación de que ambas matrices son idénticas
error_maximo = np.max(np.abs(H_numpy - H_opencv))
print(f"\nDiferencia máxima entre ambos métodos: {error_maximo:.10f}")

Homografía con OpenCV:
 [[ 0.49636364  0.10727273 10.        ]
 [-0.12545455  0.62909091 20.        ]
 [-0.00254545  0.00036364  1.        ]]

Diferencia máxima entre ambos métodos: 0.0000010221
